# Tuesday, hands on: the report you would sign
#
# > "If there is a gap, I want to know which orders and which channel."
#
# Replace every `__TODO__`. The checks tell you whether a step worked. Take the row count before
# you take any total; that is the whole discipline of the day.

In [ ]:
import pathlib
import sys

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

conn = kit.connect()
kit.flow(["attach", "count", "explain", "collapse", "report"], lit=[0], title="Where you are")

## 1. Rows before
#
# How many rows are in `orders`, and how many in `payments`? Write both down on paper before you
# run anything else.

In [ ]:
before = kit.sql("__TODO1__", conn=conn)
kit.check("both table sizes came back", len(before) == 2, str(before))

## 2. Rows after
#
# LEFT JOIN orders to payments and count. Do not sum anything yet.

In [ ]:
after = kit.sql("__TODO2__", conn=conn)[0]
n = list(after.values())[0]
kit.check("the join returned 1,450 rows", n == 1450, f"{n} rows")
kit.vflow(["1,000 orders", "LEFT JOIN payments", f"{n} rows", "explain the difference"],
          lit=[3], title="The count check")

## 3. Explain the difference
#
# How many orders carry more than one payment row, and how many carry none? Those two numbers
# plus 1,000 have to account for 1,450 exactly.

In [ ]:
multi = kit.sql("__TODO3__", conn=conn)[0]
none = kit.sql("__TODO4__", conn=conn)[0]
m, z = list(multi.values())[0], list(none.values())[0]
print(f"orders with more than one payment: {m}")
print(f"orders with no payment at all    : {z}")
kit.check("the arithmetic accounts for every row", 1000 + m == 1450, f"1000 + {m}")

## 4. Which orders were never paid
#
# Anand asked for these by name. Keep every order, then keep only the ones that failed to match.

In [ ]:
unpaid = kit.sql("__TODO5__", conn=conn)
kit.check("thirty orders were never paid", len(unpaid) == 30, f"{len(unpaid)}")
kit.ladder(["every order", "LEFT JOIN payments", "keep the NULL side", "the unpaid list"],
           lit=[2], title="An anti-join finds absence")

## 5. Retries against instalments
#
# Two payment rows with the same amount is a repeated charge. Two rows with different amounts is a
# split invoice, which is correct. A rule that deleted every duplicate would delete four hundred
# legitimate instalments.

In [ ]:
kinds = kit.sql("__TODO6__", conn=conn)
kit.table(list(kinds[0]) if kinds else ["kind"], [list(r.values()) for r in kinds],
          caption="The 450, separated")
kit.check("four hundred are instalment plans",
          any(400 in [v for v in r.values() if isinstance(v, int)] for r in kinds), str(kinds))

## 6. The number you would sign
#
# Collapse the many side to one row per order, then join. Rows out must be 1,000.

In [ ]:
real = kit.sql("__TODO7__", conn=conn)[0]
kit.check("one row in, one row out", real["rows_out"] == 1000, str(real["rows_out"]))
kit.decision_ladder(["sum over the raw join", "sum and hope", "aggregate first, then join",
                     "aggregate, join, and state the count"], cut_at=2,
                    title="What reaches Finance")

## 7. The gap, by channel
#
# Q2 only, one row per channel, carrying booked, collected and the difference. Watch what happens
# to a channel containing an unpaid order if you leave `coalesce` out.

In [ ]:
gap = kit.sql("__TODO8__", conn=conn)
kit.check("one row per channel", len(gap) == 3, f"{len(gap)} rows")
kit.table(list(gap[0]) if gap else ["channel"], [list(r.values()) for r in gap],
          caption="Q2 booked against collected")
kit.check_summary()